# Smart Fire Extinguisher Availability Monitoring
### AI-Based Facility Safety Inspection System

This notebook implements a complete safety monitoring pipeline that automatically verifies whether fire extinguishers are present and accessible at their designated locations. 

#### Core Capabilities:
1. **Red Extinguisher Blob Detection**: Uses HSV color segmentation and morphological shape filtering to locate fire extinguishers.
2. **Predefined Station Alignment**: Compares real-time detections with predefined 2D points to check if extinguishers are missing or displaced.
3. **YOLOv8 & Contour Obstacle Detection**: Integrates YOLOv8 to detect standard obstacles (people, chairs, luggage) and uses non-red contour subtraction to detect custom blockages (boxes, crates).
4. **Accessibility Zone Overlap Engine**: Computes distance intersections to identify blocked safety clearances.
5. **Auditing & Reporting**: Saves photographic evidence of violations automatically and exports a comprehensive inspection report CSV.

In [ ]:
# Setup imports
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# PyTorch 2.6 weights_only strictness monkeypatch for older YOLOv8 serialization
import torch
try:
    original_load = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return original_load(*args, **kwargs)
    torch.load = patched_load
    print("PyTorch weights_only patch applied successfully.")
except Exception as e:
    print(f"Could not apply PyTorch patch: {e}")

from ultralytics import YOLO

# Setup folder structures relative to the Code directory
os.makedirs("../Models", exist_ok=True)
os.makedirs("../Inputs", exist_ok=True)
os.makedirs("../Outputs/evidence_snapshots", exist_ok=True)
print("Project directories verified/created successfully.")

## 1. Generate Synthetic Test Video
Run this cell to create a high-fidelity synthetic test video (`../Inputs/suite_00_synthetic_simulation.mp4`) that simulates all four main safety scenarios over 15 seconds:
- **Frames 0 - 100**: Extinguisher present and clear (Safe).
- **Frames 50 - 80**: A person's shadow passes in front of the extinguisher (Blocked shadow).
- **Frames 100 - 200**: A cardboard box slides in front of the extinguisher, covering its base (Blocked box).
- **Frames 200 - 300**: The extinguisher is removed from its station (Missing).
- **Frames 300 - 450**: The extinguisher is placed at a wrong location (Displaced), and the obstacle leaves.

In [ ]:
def generate_synthetic_video(output_path):
    width, height = 640, 480
    fps = 30
    num_frames = 450
    
    fourcc = cv2.VideoWriter_fourcc(*'avc1')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    px, py = 320, 200
    ox_start, oy = -100, 220
    ox_end = 280
    
    for f in range(num_frames):
        frame = np.ones((height, width, 3), dtype=np.uint8) * 220
        cv2.line(frame, (0, 350), (640, 350), (100, 100, 100), 3) # floor
        
        # Grid lines for corridor wall
        for x in range(0, 640, 80):
            cv2.line(frame, (x, 0), (x, 350), (200, 200, 200), 1)
        for y in range(0, 350, 50):
            cv2.line(frame, (0, y), (640, y), (200, 200, 200), 1)
            
        cv2.rectangle(frame, (px - 15, py - 40), (px + 15, py + 40), (120, 120, 120), -1) # station bracket
        cv2.putText(frame, "EXTINGUISHER STATION 01", (px - 80, py - 60), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (80, 80, 80), 1)
        
        extinguisher_pos = None
        draw_obstacle = False
        ox = ox_start
        
        if 0 <= f < 100:
            extinguisher_pos = (px, py)
        elif 100 <= f < 200:
            extinguisher_pos = (px, py)
            draw_obstacle = True
            progress = (f - 100) / 100.0
            ox = int(ox_start + (ox_end - ox_start) * progress)
        elif 200 <= f < 300:
            extinguisher_pos = None
            draw_obstacle = True
            ox = ox_end
        elif 300 <= f < 450:
            extinguisher_pos = (150, 300) # Displaced
            draw_obstacle = True
            progress = (f - 300) / 150.0
            ox = int(ox_end + (700 - ox_end) * progress)
            
        # Extinguisher drawing utility
        def draw_ext(img, pos):
            x, y = pos
            cv2.rectangle(img, (x - 15, y - 35), (x + 15, y + 35), (0, 0, 220), -1) # body
            cv2.circle(img, (x, y - 35), 15, (0, 0, 220), -1) # dome
            cv2.rectangle(img, (x - 5, y - 52), (x + 5, y - 48), (30, 30, 30), -1) # valve
            cv2.line(img, (x, y - 48), (x + 12, y - 55), (30, 30, 30), 3) # handle
            cv2.line(img, (x, y - 48), (x - 15, y - 38), (30, 30, 30), 2) # hose
            cv2.circle(img, (x, y - 20), 4, (255, 255, 255), -1) # gauge
            cv2.circle(img, (x, y - 20), 4, (0, 0, 0), 1)
            cv2.rectangle(img, (x - 10, y - 10), (x + 10, y + 20), (255, 255, 255), -1) # label
            cv2.putText(img, "FIRE", (x - 8, y + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 255), 1)
            cv2.putText(img, "EXT.", (x - 8, y + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 0), 1)
            
        def draw_box(img, x, y):
            cv2.rectangle(img, (x, y), (x + 100, y + 100), (180, 110, 50), -1) # Box brown
            cv2.rectangle(img, (x, y), (x + 100, y + 100), (120, 70, 30), 3) # Border
            cv2.line(img, (x + 50, y), (x + 50, y + 100), (20, 20, 20), 8) # Tape
            cv2.putText(img, "CAUTION", (x + 10, y + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            cv2.putText(img, "OBSTACLE", (x + 10, y + 55), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            
        if extinguisher_pos is not None:
            draw_ext(frame, extinguisher_pos)
            
        if draw_obstacle:
            draw_box(frame, ox, oy)
            
        # Shadow silhouette
        if 50 < f < 80:
            px_shadow = int(100 + (f - 50) * 15)
            overlay = frame.copy()
            cv2.circle(overlay, (px_shadow, 250), 20, (150, 150, 150), -1)
            cv2.ellipse(overlay, (px_shadow, 340), (35, 70), 0, 0, 360, (150, 150, 150), -1)
            cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
            
        out.write(frame)
        
    out.release()
    print("Synthetic video successfully saved to ../Inputs/suite_00_synthetic_simulation.mp4")

generate_synthetic_video("../Inputs/suite_00_synthetic_simulation.mp4")

## 2. Interactive Coordinate Picker
To inspect a real facility image or video, you must define the `(x, y)` location of each extinguisher mounting point on the camera frame. 
Run this cell to display the first frame of the video with coordinate gridlines. You can read the `x` (horizontal) and `y` (vertical) pixel values directly from the axes.

In [ ]:
cap = cv2.VideoCapture("../Inputs/suite_00_synthetic_simulation.mp4")
ret, frame = cap.read()
cap.release()

if ret:
    # Convert BGR to RGB for matplotlib display
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(10, 7))
    plt.imshow(frame_rgb)
    plt.grid(True, color='cyan', alpha=0.5, linestyle='--')
    plt.title("Predefined Point Coordinate Picker (Hover over points or use axes)")
    plt.xlabel("X Coordinate (pixels)")
    plt.ylabel("Y Coordinate (pixels)")
    plt.show()
else:
    print("Could not load video frame. Please check the video path.")

## 3. Predefine Extinguisher Monitoring Points
Define your target mounting points below in the `predefined_stations` list. 
- You can choose to run **auto-calibration** to automatically scan the first 30 frames of the video and find the extinguisher center coordinate, or define them manually.
- `radius` is the accessibility zone clearance circle (in pixels).

In [ ]:
# Auto-Calibration Settings
auto_calibrate = True
video_path = "../Inputs/suite_00_synthetic_simulation.mp4"

predefined_stations = []

if auto_calibrate:
    print("Running auto-calibration on the input video to locate the extinguisher station...")
    cap = cv2.VideoCapture(video_path)
    centroids = []
    frame_count = 0
    while cap.isOpened() and frame_count < 30:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        lower_red1 = np.array([0, 100, 100])
        upper_red1 = np.array([10, 255, 255])
        mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
        lower_red2 = np.array([170, 100, 100])
        upper_red2 = np.array([180, 255, 255])
        mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
        red_mask = cv2.bitwise_or(mask1, mask2)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
        contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) > 150:
                x, y, w, h = cv2.boundingRect(cnt)
                if 1.0 <= float(h)/w <= 5.0:
                    centroids.append((x + w//2, y + h//2))
    cap.release()
    
    if len(centroids) > 0:
        xs = [c[0] for c in centroids]
        ys = [c[1] for c in centroids]
        cx, cy = int(np.median(xs)), int(np.median(ys))
        predefined_stations = [{
            "id": 1,
            "name": "Auto-Detected Station 01",
            "x": cx,
            "y": cy,
            "radius": 65
        }]
        print(f"Calibration successful! Station registered at ({cx}, {cy}).")
    else:
        print("Calibration failed: No extinguisher detected. Defaulting to manual settings.")
        
if not predefined_stations:
    # Manual configuration backup
    predefined_stations = [
        {
            "id": 1,
            "name": "Zone A Corridor - Station 01",
            "x": 320,
            "y": 200,
            "radius": 65
        }
    ]
    print("Using manual coordinate settings.")


## 4. Run the Safety Inspection Loop
This is the core analysis engine. It processes the video frame-by-frame:
1. **Red Extinguishers**: Segments red color using HSV bounds, runs contour extraction, and filters for cylindrical aspect ratios.
2. **YOLOv8 Obstacles (Optimized)**: Runs inference once every 10 frames to optimize CPU throughput. Intermediate frames reuse cached detections.
3. **Custom Obstacles (Dynamic Thresholding)**: Dynamically thresholds elements darker than the background (exposure-proof) and subtracts the red mask.
4. **Depth-Aware Overlap Evaluator**: Overlaps for dynamic moving objects (e.g. people) check ground contact points (feet) instead of full bounding box height to avoid false blockages when a person passes *behind* the unit.
5. **Notebook Failsafe**: Uses a robust `try...finally` block to ensure OpenCV GUI windows and capture objects are always freed during interrupts.

*An OpenCV window will open showing the live analysis stream (press **'q'** to close it). The output video is saved to `../Outputs/output_annotated.mp4`.*

In [ ]:
def run_inspection(video_path, output_path, stations):
    print("Initializing YOLOv8 model (saving weight to ../Models/yolov8n.pt)...")
    model = YOLO("../Models/yolov8n.pt")
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video file.")
        return
        
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    
    fourcc = cv2.VideoWriter_fourcc(*'avc1')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Audit log structures
    history = []
    station_states = {s["id"]: {"status": "Unknown", "frames_in_state": 0} for s in stations}
    frame_count = 0
    
    # Caching/Optimization variables
    yolo_obstacles = []
    yolo_skip_frames = 10
    
    print("Starting frame processing... (Press 'q' in the OpenCV window to stop early)")
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            frame_count += 1
            annotated_frame = frame.copy()
            hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
            
            # 1. SEGMENT RED COLOR FOR EXTINGUISHERS
            lower_red1 = np.array([0, 100, 100])
            upper_red1 = np.array([10, 255, 255])
            mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
            
            lower_red2 = np.array([170, 100, 100])
            upper_red2 = np.array([180, 255, 255])
            mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
            
            red_mask = cv2.bitwise_or(mask1, mask2)
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
            red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel)
            red_mask = cv2.morphologyEx(red_mask, cv2.MORPH_CLOSE, kernel)
            
            red_contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            detected_extinguishers = []
            
            for cnt in red_contours:
                area = cv2.contourArea(cnt)
                if area < 150:
                    continue
                x, y, w, h = cv2.boundingRect(cnt)
                aspect_ratio = float(h) / w
                # Fire extinguisher cylindrical check
                if 1.0 <= aspect_ratio <= 5.0:
                    centroid = (x + w // 2, y + h // 2)
                    detected_extinguishers.append({
                        "bbox": (x, y, w, h),
                        "centroid": centroid
                    })
                    # Draw red extinguisher bounding box
                    cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), (0, 0, 255), 2)
                    cv2.putText(annotated_frame, "Extinguisher", (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                    
            # 2. RUN YOLOv8 FOR STANDARD OBSTACLES (Optimized to run every N frames)
            if (frame_count - 1) % yolo_skip_frames == 0:
                results = model(frame, verbose=False)
                yolo_obstacles = []
                for box in results[0].boxes:
                    cls_id = int(box.cls[0])
                    label = model.names[cls_id]
                    conf = float(box.conf[0])
                    
                    if label in ["person", "chair", "couch", "backpack", "suitcase", "handbag", "bottle"]:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])
                        yolo_obstacles.append({
                            "bbox": (x1, y1, x2 - x1, y2 - y1),
                            "label": f"{label} ({conf:.2f})",
                            "class": label
                        })
            
            # Draw YOLO bounding boxes in every frame (using cached obstacles)
            for obs in yolo_obstacles:
                x1, y1, w, h = obs["bbox"]
                label = obs["label"]
                class_name = obs["class"]
                cv2.rectangle(annotated_frame, (x1, y1), (x1 + w, y1 + h), (255, 100, 0), 2)
                cv2.putText(annotated_frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 100, 0), 1)
                    
            # 3. SEGMENT CUSTOM OBSTACLES (Dynamic Thresholding & red subtraction)
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            # Adaptively compute brightness background threshold (exposure-proof)
            avg_val = np.mean(gray)
            thresh_val = int(avg_val * 0.9)
            _, thresh = cv2.threshold(gray, thresh_val, 255, cv2.THRESH_BINARY_INV)
            # Subtract red extinguisher mask
            obstacle_mask = cv2.subtract(thresh, red_mask)
            obstacle_mask = cv2.morphologyEx(obstacle_mask, cv2.MORPH_OPEN, kernel)
            obstacle_mask = cv2.morphologyEx(obstacle_mask, cv2.MORPH_CLOSE, kernel)
            
            general_contours, _ = cv2.findContours(obstacle_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contour_obstacles = []
            for cnt in general_contours:
                area = cv2.contourArea(cnt)
                if area < 300: # ignore minor details
                    continue
                x, y, w, h = cv2.boundingRect(cnt)
                contour_obstacles.append({
                    "bbox": (x, y, w, h),
                    "contour": cnt
                })
                
            # 4. ANALYZE PREDEFINED STATIONS (Depth/Perspective-Aware)
            displaced_detected = []
            active_alert = False
            
            for st in stations:
                st_id = st["id"]
                px, py, pr = st["x"], st["y"], st["radius"]
                
                # Find if extinguisher is within radius of station
                ext_match = None
                for ext in detected_extinguishers:
                    ex, ey = ext["centroid"]
                    dist = np.sqrt((ex - px)**2 + (ey - py)**2)
                    if dist <= pr:
                        ext_match = ext
                        break
                        
                status = "Missing"
                blocker_info = None
                
                if ext_match is not None:
                    status = "Present & Clear"
                    
                    # Check YOLO obstacle overlaps (Depth-Aware logic)
                    for obs in yolo_obstacles:
                        ox, oy, ow, oh = obs["bbox"]
                        
                        # Depth Check: For a person, check ground contact point (feet)
                        if obs["class"] == "person":
                            cx = ox + ow // 2
                            cy = oy + oh # feet position
                            dist_to_zone = np.sqrt((cx - px)**2 + (cy - (py + 40))**2)
                            # If their feet are not inside the safety radius projection, ignore them
                            if dist_to_zone <= pr:
                                status = "Blocked"
                                blocker_info = obs["label"]
                                cv2.line(annotated_frame, (px, py), (cx, cy), (0, 165, 255), 2)
                                break
                        else:
                            # Standard closest-point approximation for general objects
                            cx = max(ox, min(px, ox + ow))
                            cy = max(oy, min(py, oy + oh))
                            if np.sqrt((cx - px)**2 + (cy - py)**2) <= pr:
                                status = "Blocked"
                                blocker_info = obs["label"]
                                cv2.line(annotated_frame, (px, py), (cx, cy), (0, 165, 255), 2)
                                break
                            
                    # Check custom contour obstacle overlaps
                    if status == "Present & Clear":
                        for obs in contour_obstacles:
                            ox, oy, ow, oh = obs["bbox"]
                            cx = max(ox, min(px, ox + ow))
                            cy = max(oy, min(py, oy + oh))
                            is_bracket = (abs(ox - (px - 15)) < 10 and abs(oy - (py - 40)) < 10)
                            if np.sqrt((cx - px)**2 + (cy - py)**2) <= pr and not is_bracket and not (oy < py - 40):
                                status = "Blocked"
                                blocker_info = "Box/Obstacle"
                                cv2.rectangle(annotated_frame, (ox, oy), (ox + ow, oy + oh), (0, 165, 255), 2)
                                cv2.putText(annotated_frame, "Blockage", (ox, oy - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 165, 255), 1)
                                cv2.line(annotated_frame, (px, py), (cx, cy), (0, 165, 255), 2)
                                break
                else:
                    # Check for displaced extinguishers elsewhere
                    displaced_exts = []
                    for ext in detected_extinguishers:
                        ex, ey = ext["centroid"]
                        near_any = False
                        for s_pt in stations:
                            if np.sqrt((ex - s_pt["x"])**2 + (ey - s_pt["y"])**2) <= s_pt["radius"]:
                                near_any = True
                                break
                        if not near_any:
                            displaced_exts.append(ext)
                            
                    if len(displaced_exts) > 0:
                        status = "Missing & Displaced"
                        # Draw line pointing to displaced position
                        for de in displaced_exts:
                            dex, dey = de["centroid"]
                            cv2.line(annotated_frame, (px, py), (dex, dey), (0, 0, 255), 1, cv2.LINE_AA)
                            cv2.circle(annotated_frame, (dex, dey), 5, (0, 0, 255), -1)
                            cv2.putText(annotated_frame, "DISPLACED", (dex - 30, dey - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                    else:
                        status = "Missing"
                
                # Draw zone ring overlay
                if status == "Present & Clear":
                    ring_color = (0, 200, 0) # Green
                elif status == "Blocked":
                    ring_color = (0, 165, 255) # Orange
                    active_alert = True
                else:
                    ring_color = (0, 0, 255) # Red
                    active_alert = True
                    
                # Overlay safety zone
                cv2.circle(annotated_frame, (px, py), pr, ring_color, 2, lineType=cv2.LINE_AA)
                cv2.circle(annotated_frame, (px, py), 4, ring_color, -1)
                cv2.putText(annotated_frame, f"{st['name']}: {status}", (px - 90, py + pr + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.35, ring_color, 1)
                if blocker_info:
                    cv2.putText(annotated_frame, f"Obstacle: {blocker_info}", (px - 90, py + pr + 27), cv2.FONT_HERSHEY_SIMPLEX, 0.35, ring_color, 1)
                
                # Log status change with buffer to prevent noise
                old_state = station_states[st_id]["status"]
                if status != old_state:
                    station_states[st_id]["frames_in_state"] += 1
                    # Require 5 consecutive frames in new state to confirm change
                    if station_states[st_id]["frames_in_state"] >= 5:
                        station_states[st_id]["status"] = status
                        station_states[st_id]["frames_in_state"] = 0
                        
                        # Record in log list
                        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
                        log_entry = {
                            "Timestamp": timestamp,
                            "Frame": frame_count,
                            "Station_ID": st_id,
                            "Station_Name": st["name"],
                            "Status": status,
                            "Details": f"Blocked by {blocker_info}" if status == "Blocked" else ("Extinguisher displaced elsewhere" if status == "Missing & Displaced" else "None")
                        }
                        history.append(log_entry)
                        print(f"[{timestamp}] Frame {frame_count:03d} -> Station '{st['name']}' status changed to [{status}]")
                        
                        # Capture evidence snapshot if violation occurs
                        if status in ["Blocked", "Missing", "Missing & Displaced"]:
                            ev_filename = f"../Outputs/evidence_snapshots/violation_st{st_id}_f{frame_count}_{status.replace(' & ', '_')}.jpg"
                            cv2.imwrite(ev_filename, annotated_frame)
                else:
                    station_states[st_id]["frames_in_state"] = 0
                    
            # HUD Info Text overlays
            cv2.rectangle(annotated_frame, (10, 10), (320, 65), (50, 50, 50), -1) # Background bar
            cv2.putText(annotated_frame, f"FRAME: {frame_count:03d} | STATE MONITOR", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1, cv2.LINE_AA)
            status_counts = {"Safe": 0, "Blocked": 0, "Violation": 0}
            for st_id, details in station_states.items():
                if details["status"] == "Present & Clear": status_counts["Safe"] += 1
                elif details["status"] == "Blocked": status_counts["Blocked"] += 1
                else: status_counts["Violation"] += 1
                
            cv2.putText(annotated_frame, f"ACCESSIBLE: {status_counts['Safe']} | BLOCKED: {status_counts['Blocked']} | MISSING: {status_counts['Violation']}", 
                        (15, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (100, 255, 100), 1, cv2.LINE_AA)
                        
            # Flashing Warning Alerts on HUD
            if active_alert:
                pulse_speed = 10
                text_val = "CRITICAL: SAFETY VIOLATION DETECTED"
                if (frame_count // pulse_speed) % 2 == 0:
                    cv2.putText(annotated_frame, text_val, (15, 57), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1, cv2.LINE_AA)
                else:
                    cv2.putText(annotated_frame, text_val, (15, 57), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (150, 150, 255), 1, cv2.LINE_AA)
            else:
                cv2.putText(annotated_frame, "STATUS: SYSTEM CLEAR", (15, 57), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 0), 1, cv2.LINE_AA)
                
            out.write(annotated_frame)
            cv2.imshow("AI Facility Safety Inspection - Fire Extinguisher Monitor", annotated_frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Processing cancelled by user.")
                break
    finally:
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        print(f"Processing finished. Total frames processed: {frame_count}")
    return history

inspection_log = run_inspection("../Inputs/suite_00_synthetic_simulation.mp4", "../Outputs/suite_00_annotated.mp4", predefined_stations)

## 5. Export Inspection Log & Render Audit Summary
Export the safety events collected during inspection into `../Outputs/suite_inspection_report.csv` and visualize the metrics.

In [ ]:
if len(inspection_log) > 0:
    df = pd.DataFrame(inspection_log)
    # Save report to CSV
    df.to_csv("../Outputs/suite_inspection_report.csv", index=False)
    print("Inspection report successfully saved to ../Outputs/suite_inspection_report.csv")
    
    # Display the logs
    display(df)
    
    # Plot the states transitions chronologically
    plt.figure(figsize=(12, 5))
    colors = {"Present & Clear": "green", "Blocked": "orange", "Missing": "red", "Missing & Displaced": "darkred"}
    
    for i, st in enumerate(predefined_stations):
        st_df = df[df["Station_ID"] == st["id"]]
        if not st_df.empty:
            plt.step(st_df["Frame"], st_df["Status"], where='post', label=st["name"], color='blue', linewidth=2)
            
            # Scatter markers to highlight each status transition point
            for idx, row in st_df.iterrows():
                plt.scatter(row["Frame"], row["Status"], color=colors.get(row["Status"], "blue"), s=100, zorder=5)
                
    plt.title("Extinguisher Safety Status Transitions over Time")
    plt.xlabel("Video Frame Number")
    plt.ylabel("Safety Status")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No state changes were recorded. The extinguisher remained in its initial state throughout the video.")